# 03 - Evaluasi ROUGE & Perbandingan Waktu Proses (MP4 vs MP3 vs WAV)

Mengevaluasi performa model IndoT5 dan membandingkan waktu transkripsi Whisper antar format file.

## 1. Import

In [ ]:
import pandas as pd
import evaluate
import time
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import matplotlib.pyplot as plt

## 2. Load Model Fine-tuned

In [ ]:
MODEL_DIR = Path('../models/indot5_finetuned')
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = model.to(device).eval()

df_test = pd.read_csv('../dataset/03_paired/test.csv')
df_test.head()

## 3. Fungsi Prediksi

In [ ]:
def predict(text: str, max_new_tokens: int = 150) -> str:
    inp = tokenizer(
        'ringkas: ' + text,
        return_tensors='pt',
        max_length=1024,
        truncation=True,
    ).to(device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

## 4. Hitung Skor ROUGE

In [ ]:
rouge = evaluate.load('rouge')
preds = [predict(s) for s in df_test['source']]
results = rouge.compute(predictions=preds, references=df_test['target'].tolist())
for k, v in results.items():
    print(f'{k}: {v:.4f}')

## 5. Perbandingan Waktu Transkripsi per Format File

In [ ]:
import sys
sys.path.append('..')
from modules.transcriber import transcribe_audio

FILES = {
    'MP4': Path('../dataset/01_raw/video_mp4/rapat_01.mp4'),
    'MP3': Path('../dataset/01_raw/audio_mp3/rapat_01.mp3'),
    'WAV': Path('../dataset/01_raw/audio_wav/rapat_01.wav'),
}

hasil_waktu = {}
for label, fp in FILES.items():
    if not fp.exists():
        print(f'[SKIP] {fp} tidak ditemukan')
        continue
    t0 = time.time()
    transcribe_audio(fp)
    hasil_waktu[label] = round(time.time() - t0, 2)
    print(f'{label}: {hasil_waktu[label]} detik')

df_waktu = pd.DataFrame(list(hasil_waktu.items()), columns=['Format', 'Waktu (detik)'])
df_waktu

## 6. Visualisasi Perbandingan Waktu

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(
    df_waktu['Format'],
    df_waktu['Waktu (detik)'],
    color=['#4A90D9', '#E67E22', '#2ECC71'],
)
plt.title('Perbandingan Waktu Transkripsi per Format File')
plt.ylabel('Waktu (detik)')
plt.xlabel('Format Input')
plt.tight_layout()
plt.savefig('../dataset/perbandingan_waktu.png', dpi=150)
plt.show()